# T4. Trades: sizing the fleet

**The question:** three missions share one airframe catalog. Which bird wins
each mission, and does any one bird do everything?

The model answers through `longeron.analysis`. The `trades` module scores
every discrete mix through the interpreter. The `mdao` module sizes the
continuous variables with OpenMDAO. The interpreter stays the single source
of semantics, so the notebook checks every solver result against the model
itself.

**You will learn how to:**

- read the catalog and its requirements straight from the model;
- score every discrete mix exactly and read one Pareto front per mission;
- brush the mission space in linked widgets, up to the compromise dashboard;
- size the ISR winner with OpenMDAO, then read the N2 map and the margins;
- swap a declared external aerodynamics analysis in with one keyword;
- write the analysis results back into the model.

**Prerequisites:** T2 for execution and T3 for the review widgets. Install
the analysis extras with `pip install "longeron[trades,mdao,viz]"`. The
widgets need JupyterLab. On a static page each widget shows a placeholder.

In [ ]:
import longeron
from longeron.analysis import mdao, trades, viz

model = longeron.load("../examples/uav_missions.sysml")
missions = {
    "ISR": ("UavMissions::IsrUav", "stationMinutes"),
    "logistics": ("UavMissions::LogisticsUav", "payloadRangeKgKm"),
    "intercept": ("UavMissions::InterceptUav", "maxTargetSpeed"),
}
studies = {name: trades.TradeStudy(model, qname) for name, (qname, _) in missions.items()}
for name, study in studies.items():
    points = ", ".join(f"{p.name}[{len(p.variants)}]" for p in study.points.values())
    print(f"{name:9s} -> {points}")

## The destination first: the compromise dashboard

The finished artifact opens the notebook.
`analysis.dashboard.mission_dashboard` bakes 2016 interpreter-exact mixes
into one candidate table. Four linked panels share one 1080p screen:

- a header strip with the `Pareto only` toggle and the lineup-size slider;
- the parallel-coordinates table beside the MOE-versus-cost scatter;
- one tab set: a summary tab for all three missions, then one tab per
  mission with its requirement sliders and margin card;
- the to-scale 3D lineup beside the tab set, so slider moves and their
  shapes share one glance.

Requirement sliders read their defaults from the model's own requirement
attributes. Priority sliders sit in the summary tab and feed the MOE, the
documented compromise score. The `Pareto only` toggle hides dominated
candidates from every panel. One candidate dominates another when it costs
no more and scores at least as well on every mission metric. Priority
weights never change the front.

Drag the `intercept` priority to 100 and the dart takes the star. Hand the
weight back to `ISR` and the tail-sitter returns. The rest of this notebook
rebuilds this surface one piece at a time.

*(Widget cell: captured at landing.)*

In [ ]:
from longeron.analysis import dashboard

dash = dashboard.mission_dashboard(model)
dash

## The catalog: seven variation points, three missions

`examples/uav_missions.sysml` models one component catalog.
`UavMissions::Catalog` holds seven variation points: airframe, motors,
props, battery, sensor, cargo bay, and structural material. A **mix** is one
choice at every variation point that a mission uses. Three mission contexts
evaluate the shared catalog:

- `IsrUav` loiters on station with a stabilized sensor. Its metric is
  `stationMinutes`.
- `LogisticsUav` flies a parcel out and returns with an empty bay. Its
  metric is `payloadRangeKgKm`.
- `InterceptUav` dashes one way to catch a crossing target. Its metric is
  `maxTargetSpeed`.

The four airframes are different machines:

- `boxQuad` is a cheap rotor-only quad on an open frame.
- `teardropQuad` carries the same four rotors inside a lathed low-drag
  shell.
- `vtolWing` hovers on four props and cruises on a 2.6 m wing.
- `dartInterceptor` is a rail-launched pusher with almost no payload.

The airframes are fictional; everything bolted to them is a real
commercial part with nominal catalog figures. The motor tiers are the
T-Motor Antigravity MN4006, the SunnySky X4112S, and the 2 kW T-Motor
AT4120. The props are APC electrics plus a T-Motor 15-inch carbon
lifter. The battery axis carries three Tattu LiPo packs and one 18650
li-ion pack, so pack chemistry is a real trade: the li-ion pack holds
the most energy per kilogram and a tenth of the discharge ceiling.

All the physics lives in `calc def` bodies that the interpreter evaluates
directly. The calc definitions sit in four discipline packages:
`Aerodynamics`, `Propulsion`, `Structures`, and `Performance`. That
organization returns later, when the MDAO bridge groups the generated
problem by these same packages. The diagram below draws the variation tree.
The same variation points return as brushable columns in every
parallel-coordinates view.

*(Widget cell: captured at landing.)*

In [ ]:
try:
    from longeron import diagrams

    display(diagrams.structure_diagram(model.find("UavMissions::Catalog"), show_attributes=False))
except ImportError:
    print("optional: pip install -e vendor/ipyelk enables the SysML diagrams -- skipping here")

## The requirements, as the model states them

`UavMissions::MissionRequirements` states each tasking as a requirement
definition. Each requirement carries its subject, its `assume` clause, and
its `require` clause. The numeric floors are model attributes:
`minStationMinutes`, `minPayloadKg`, `minDeliveryRadiusKm`, and
`targetSpeed`. The dashboard sliders read their defaults from these same
attributes, so the review surface and the requirements cannot drift apart.

*(Widget cell: captured at landing.)*

In [ ]:
try:
    from longeron import diagrams

    display(diagrams.structure_diagram(model.find("UavMissions::MissionRequirements")))
except ImportError:
    print("optional: pip install -e vendor/ipyelk enables the SysML diagrams -- skipping here")

## The honest solver choice

CP-SAT works on fixed-point integer arithmetic. The mapper inlines `calc`
invocations, encodes `max()` and `min()` natively, and unrolls constant
integer exponents. That covers the shared `MissionUAV` platform: structural
sizing, the mass and cost build-ups, and both compatibility constraints.
The cell below enumerates the platform with CP-SAT and checks the result
against the interpreter, mix for mix.

The mission layers stay out of reach. `HoverPower` raises mass to the power
1.5, `DashSpeed` takes a cube root, and the cruise attributes branch on
wing span. No fixed-point encoding is exact for these forms. The mapper
refuses each mission with a one-line verdict that names the innermost
operation. The refusal is a design decision. A wrong encoding would return
wrong fronts silently.

Exhaustion is the honest alternative at this scale. `all_architectures()`
walks the whole mix space through the interpreter, exactly, in under a
second. Each infeasible mix carries `violations`, the names of the
constraints it breaks.

In [ ]:
platform = trades.TradeStudy(model, "UavMissions::MissionUAV")
solved = {tuple(sorted(a.selection.items())) for a in platform.enumerate()}
exact = {tuple(sorted(a.selection.items())) for a in platform.all_architectures() if a.verified}
assert solved == exact  # the interpreter is the oracle; CP-SAT must agree
print(f"platform  {len(solved):3d} of 288 shared mixes feasible (CP-SAT == interpreter)\n")

for name, study in studies.items():
    try:
        study.enumerate()
    except longeron.analysis.AnalysisError as err:
        print(f"{name:9s} {err}\n")

spaces = {name: study.all_architectures() for name, study in studies.items()}
for name, archs in spaces.items():
    feasible = sum(a.verified for a in archs)
    print(f"{name:9s} {feasible:3d} of {len(archs)} mixes feasible (interpreter)")

## Where the drag numbers come from

No airframe quotes its drag area by fiat. Every `dragArea` value is a
wetted-area buildup inside the model. The buildup sums skin friction times
wetted area times a form factor over every surface. It then adds 15% for
interference and a bluff-body term for the open frame and exposed rotor
gear. Big wings pay for their extra skin. Slender bodies earn their
advantage from the same arithmetic.

In [ ]:
airframes = studies["ISR"].points["airframe"]
stories = {
    "boxQuad": "all bluff: open frame, battery, motor cans",
    "teardropQuad": "skinned lathe l/d 4.8 + bluff arms/motors",
    "vtolWing": "fuselage + BOTH wing pairs + 4 wingtip pods",
    "dartInterceptor": "slender body l/d 11 + thin wing + fins",
}
print(f"{'airframe':18s}{'CdA m^2':>9s}   drag story (the model's own buildup)")
for name, variant in airframes.variants.items():
    print(f"{name:18s}{variant['dragArea']:9.4f}   {stories[name]}")
print("\nTakeaway: the skin you fly is the drag you pay -- the dart's 11:1 body")
print("undercuts the teardrop 2:1, and both embarrass the open frame 4:1 and up.")

## Mission 1, ISR: the wing buys the loiter

On station, a quad must hover. Momentum theory prices hover power from disk
loading. A winged family flies slow on wing lift instead, at less than one
tenth of hover power.

Read the figure as a staircase. The marked points are the **front**, the
mixes that no other mix beats on both cost and endurance. Budget quads hold
the cheap corner at 25 to 48 minutes. The winged VTOL takes over from there
and runs past three hours, and the pack chemistry decides the top step: the
li-ion pack's extra watt-hours buy the last 47 minutes. The interceptor is
absent, because its 0.6 kg bay cannot carry the gimbal the mission
requires. The pale crosses are
infeasible mixes, and the figure plots each cross at the score its broken
constraints deny it.

In [ ]:
isr_front = trades.pareto(
    [a for a in spaces["ISR"] if a.verified],
    minimize=("missionCost",),
    maximize=("stationMinutes",),
)
isr_best = max(isr_front, key=lambda a: a.metrics["stationMinutes"])
isr_cheap = min(isr_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["ISR"],
    x="missionCost",
    y="stationMinutes",
    sense=("min", "max"),
    panel_y="missionMass",
    xlabel="mission cost (USD)",
    ylabel="time on station (min)",
    panel_ylabel="mission mass (kg)",
    annotate={"winged VTOL, li-ion pack: 209 min": isr_best, "budget quad corner": isr_cheap},
    title="The winged VTOL owns endurance; quads keep the cheap corner",
)

## Mission 2, logistics: out heavy, back empty

The delivery flight is asymmetric. The outbound leg carries the parcel. The
return leg flies with an empty bay. A fixed hover budget covers takeoff,
drop-off, and landing. The metric multiplies the parcel mass by the
sustained radius.

Read the figure for the same family split. Rotor-borne cruise never escapes
hover power, so the quad's radius stays under 16 km with the smallest
parcel. The winged VTOL turns the same battery packs into 30 to 163 kg km
of payload-range, and the parcel lift wants LiPo watts: the delivery winner
flies the 16 Ah Tattu, not the li-ion pack. Every kilogram flies out and
back, so the carbon spar's saved grams show up here too.

In [ ]:
log_front = trades.pareto(
    [a for a in spaces["logistics"] if a.verified],
    minimize=("missionCost",),
    maximize=("payloadRangeKgKm",),
)
log_best = max(log_front, key=lambda a: a.metrics["payloadRangeKgKm"])
log_cheap = min(log_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["logistics"],
    x="missionCost",
    y="payloadRangeKgKm",
    sense=("min", "max"),
    panel_y="deliveryRadiusKm",
    xlabel="mission cost (USD)",
    ylabel="payload x radius (kg km)",
    panel_ylabel="radius (km)",
    annotate={"winged VTOL: 4 kg out to 41 km": log_best, "$1236 quad: 1 kg to 8 km": log_cheap},
    title="Wings turn batteries into payload-range; quads just clear 8 km",
)

## Mission 3, intercept: low drag wins the dash

Parasite drag limits dash speed. The catchable target speed inverts the
intercept triangle at the battery-limited dash duration. The catalog fields
two dash designs. The `dartInterceptor` pairs the lowest drag area with a
single pusher motor. The `teardropQuad` pushes twice that drag area with
four motors.

Read the figure top down. The dart owns the top of the front. The teardrop
holds the mid-price region. Plain quads catch slow crossers cheaply. Every
mix on this front flies LiPo and aluminum: no li-ion pack can feed the
AT4120's 2 kW draw, and dash physics never rewards the carbon spar's lower
mass.

In [ ]:
int_front = trades.pareto(
    [a for a in spaces["intercept"] if a.verified],
    minimize=("missionCost",),
    maximize=("maxTargetSpeed",),
)
int_best = max(int_front, key=lambda a: a.metrics["maxTargetSpeed"])
int_cheap = min(int_front, key=lambda a: a.metrics["missionCost"])
fig = viz.pareto_figure(
    spaces["intercept"],
    x="missionCost",
    y="maxTargetSpeed",
    sense=("min", "max"),
    panel_y="dashSpeed",
    xlabel="mission cost (USD)",
    ylabel="max catchable target speed (m/s)",
    panel_ylabel="dash speed (m/s)",
    annotate={"2 kW dart: 73 m/s targets": int_best, "$1272 quad catches 28 m/s": int_cheap},
    title="The dart leads the dash; the wingless teardrop takes the mid-price front",
)

## Across missions: does any one bird do everything?

Project each front onto the choices the missions share: airframe, motors,
props, battery, and material. Call that tuple the **base mix**. Several base
mixes sit on both the ISR front and the logistics front. The strongest is
the winged VTOL with X4112S motors and 11x5.5 props. Buy that bird once and
re-fit the payload bay between sorties. No base mix reaches all three
fronts, because the interceptor's dash physics is a different aircraft.

In [ ]:
def base_mix(arch):
    keep = ("airframe", "motors", "props", "battery", "material")
    return tuple(arch.selection[k] for k in keep)


fronts = {"ISR": isr_front, "logistics": log_front, "intercept": int_front}
membership = {}
for name, front in fronts.items():
    for arch in front:
        membership.setdefault(base_mix(arch), set()).add(name)
print(f"{'airframe':16s}{'motors':13s}{'props':12s}{'battery':10s}{'material':13s} fronts")
for mix, names in sorted(membership.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(
        "".join(f"{part:13s}" if i else f"{part:16s}" for i, part in enumerate(mix))
        + "  "
        + ", ".join(sorted(names))
    )
assert not any(len(names) == 3 for names in membership.values())

## Brushing the mission space

`viz.parcoords` draws one line per base mix, material included. Each base
mix is scored on every mission at once. Each metric column holds the best
value that the base mix achieves over its equipment options. The column
holds 0 where no equipment option is feasible. Dashed gray lines fail every
mission.

Brush `stationMinutes` high and `maxTargetSpeed` high. No line survives both
brushes, which restates the answer above.

*(Widget cell: captured at landing.)*

In [ ]:
cross_rows = []
cross_mixes = []
for arch in spaces["intercept"]:  # the 288 shared base mixes
    mix = dict(
        zip(("airframe", "motors", "props", "battery", "material"), base_mix(arch), strict=True)
    )
    row = dict(mix)
    for name, metric in (
        ("ISR", "stationMinutes"),
        ("logistics", "payloadRangeKgKm"),
        ("intercept", "maxTargetSpeed"),
    ):
        best = [a for a in spaces[name] if a.verified and base_mix(a) == base_mix(arch)]
        row[metric] = max((a.metrics[metric] for a in best), default=0.0)
    row["cost"] = arch.metrics["missionCost"]
    row["feasible"] = any(
        row[m] > 0 for m in ("stationMinutes", "payloadRangeKgKm", "maxTargetSpeed")
    )
    cross_rows.append(row)
    cross_mixes.append(arch)

pc = viz.parcoords(
    cross_rows,
    axes=[
        "airframe",
        "motors",
        "props",
        "battery",
        "material",
        "cost",
        "stationMinutes",
        "payloadRangeKgKm",
        "maxTargetSpeed",
    ],
)
pc

## Shapes to scale: the 3D viewer

`analysis.geometry` bakes each family parametrically from the selected
catalog values. The bake uses stdlib math only, with no CAD kernel.
`viewer3d.mesh_viewer` renders the baked mesh at full cell width. Drag to
orbit, right-drag to pan, scroll to zoom, and double-click to re-fit.

The second cell links the widgets. A traitlet observer watches the
parallel-coordinates `selected` list and re-bakes the first surviving base
mix into the viewer. Brush the `airframe` axis through its categories and
watch the shape switch family. T3 teaches the selection seam behind this
pattern.

*(Widget cells: captured at landing.)*

In [ ]:
from longeron.analysis import geometry, viewer3d

viewer3d.mesh_viewer(
    geometry.mission_geometry(studies["ISR"], isr_best),
    label=(
        f"ISR winner in hover attitude -- {isr_best.metrics['stationMinutes']:.0f} min on station"
    ),
)

In [ ]:
import json

linked = viewer3d.mesh_viewer(
    geometry.mission_geometry(studies["intercept"], cross_mixes[0]),
    label=" / ".join(base_mix(cross_mixes[0])),
)


def show_first_selected(change):
    indices = json.loads(change["new"] or "[]")
    if indices:
        mix = cross_mixes[indices[0]]
        linked.mesh_json = json.dumps(geometry.mission_geometry(studies["intercept"], mix))
        linked.label = " / ".join(base_mix(mix))


pc.observe(show_first_selected, names="selected")
linked

## Continuous sizing: how fast should the ISR winner loiter?

The trade study picked the components. `analysis.mdao` sizes what stays
continuous. `UavMissions::IsrPrime` freezes the ISR winner's mix as a
concrete part definition and leaves `loiterSpeed` free.

`mdao.build_problem` mirrors the part onto an OpenMDAO `Problem`. Attributes
become components that evaluate through the interpreter. Constraints and the
`IsrStation` requirement become margin outputs. The builder groups the
components by the calc definitions' owning packages. The model's structure
is the problem's structure.

In [ ]:
build = mdao.build_problem(
    model, "UavMissions::IsrPrime", requirements=("UavMissions::IsrStation",)
)
p = build.problem
p.run_model()
for discipline, attrs in build.disciplines.items():
    print(f"{discipline:14s} {', '.join(attrs)}")
print("loiterPowerW:  ", round(float(p.get_val("loiterPowerW")[0]), 1))
print("stationMinutes:", round(float(p.get_val("stationMinutes")[0]), 1))
p.set_val("loiterSpeed", 21.0)  # what-if: loiter at transit speed
p.run_model()
print(
    "at 21 m/s:     ",
    round(float(p.get_val("stationMinutes")[0]), 1),
    "min -- stationFloor margin",
    round(float(p.get_val("stationFloor_margin")[0]), 1),
)
p.set_val("loiterSpeed", 15.0)
p.run_model()

### The problem's shape: an N2 map

Look at the structure before you trust the numbers.
`analysis.structure.n2_view` draws the built problem as an N2 matrix in the
OpenMDAO convention. Components sit on the diagonal in execution order. Each
coupling sits in its source's row and its target's column, so feed-forward
fills the upper triangle. The dashed outlines are the discipline groups that
`build_problem` derived from the model's calc packages.

This sizing chain is pure feed-forward, so every dot sits above the
diagonal. A feedback coupling would land below the diagonal. Hover a dot to
list the coupled variables.

*(Widget cell: captured at landing.)*

In [ ]:
from longeron.analysis import structure

structure.n2_view(build)

### The margin picture

Sweep `loiterSpeed` past the legal window and every wall shows at once. Only
the walls are shaded. Wherever any margin goes negative, the band is hatched
and labeled with every constraint that binds there. The unshaded middle is
the feasible corridor.

Read the corridor edge by edge. Below about 11 m/s the `aboveStall` floor
binds. Past about 22 m/s the 90 minute `stationFloor` breaks. Beyond
24 m/s the `belowCruise` ceiling adds on top of the broken `stationFloor`.
The first-order drag polar rewards slower flight, so the best legal loiter
sits at the `aboveStall` floor.

In [ ]:
fig = viz.margin_sweep_figure(
    p,
    "loiterSpeed",
    [9.0 + 0.35 * i for i in range(50)],
    build.constraints,
    xlabel="loiter speed (m/s)",
    title="The station floor caps loiter at ~22 m/s; stall and transit limits frame the corridor",
)

## Declared external analyses: swapping the aerodynamics fidelity

First-order physics belongs in the model as `calc def` bodies.
Higher-fidelity tools live outside SysML. The convention shipped with the
example makes the model declare the binding:

```sysml
metadata def ExternalAnalysis { attribute component : String; }

calc def CruisePower {
    @ExternalAnalysis { component = "uav_aero:CruisePowerPolar"; }
    in massKg : Real;  in speed : Real;  ...
    return : Real = ...first-order drag polar...;
}
```

The calc's `in` and `return` parameters are the I/O contract.
`build_problem` validates the contract against the wrapped component's
actual inputs and outputs. The keyword `fidelity={"CruisePower": "external"}`
swaps the interpreter-backed body for the external component. Here the
component is `examples/uav_aero.py`, a synthetic polar that models Reynolds
effects and stall. Everything else in the problem stays untouched, so the
comparison below is one keyword.

In [ ]:
import sys

import matplotlib.pyplot as plt

if "../examples" not in sys.path:
    sys.path.insert(0, "../examples")  # the uav_aero entry point

lo = mdao.build_problem(model, "UavMissions::IsrPrime")
hi = mdao.build_problem(model, "UavMissions::IsrPrime", fidelity={"CruisePower": "external"})
print("bound externals:", hi.externals)

speeds = [11.0 + 0.2 * i for i in range(51)]
station = {}
for name, b in (("first-order calc body", lo), ("uav_aero polar (external)", hi)):
    values = []
    for v in speeds:
        b.problem.set_val("loiterSpeed", v)
        b.problem.run_model()
        values.append(float(b.problem.get_val("stationMinutes")[0]))
    station[name] = values

fig, ax = plt.subplots(figsize=(7.0, 3.6), layout="constrained")
for (name, values), color in zip(station.items(), ("#2f6b8f", "#c2603e"), strict=True):
    ax.plot(speeds, values, color=color, linewidth=1.6)
    best = max(range(len(speeds)), key=lambda i: values[i])
    ax.plot(speeds[best], values[best], "o", color=color, markersize=5)
    ax.annotate(
        f"{name}\nbest {values[best]:.0f} min @ {speeds[best]:.1f} m/s",
        (speeds[best], values[best]),
        xytext=(10, -6),
        textcoords="offset points",
        fontsize=8,
        color=color,
    )
ax.set_xlabel("loiter speed (m/s)")
ax.set_ylabel("time on station (min)")
ax.set_title(
    "The Reynolds/stall-aware polar backs loiter off the stall and costs 40 min",
    fontsize=10,
    loc="left",
    color="#2b2d31",
)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#d9dbdf", linewidth=0.5)

The first-order body rewards slower flight all the way to the stall floor.
The external polar prices the drag rise near stall, so its optimum backs
off to about 12 m/s. The promised endurance drops from 300 to 246
minutes. One keyword changed the physics. Nothing else moved.

## Write the answer back into the model

Analysis that stays in a notebook evaporates. The model is the source of
truth, so the model absorbs what the analysis learned. `Interpreter.snapshot`
turns a computed instance back into model elements with bound values. The
cell below runs the full loop: instantiate the design at the analyzed
loiter speed, snapshot it into the package, save, and reload.

The saved numbers deserve one clarification. The snapshot evaluates through
the model's own calc bodies, so the saved `stationMinutes` is the
first-order 280 minutes at 12.0 m/s. The polar's 246 minutes enters the
model only when a higher-fidelity calc body replaces the first-order one.
Tutorials T6 and T9 reuse this seam. They read results that the model
already carries.

In [ ]:
import math
import tempfile
from pathlib import Path

polar = station["uav_aero polar (external)"]
best_loiter = speeds[max(range(len(speeds)), key=lambda i: polar[i])]

interp = longeron.Interpreter(model)
sized = interp.instantiate("UavMissions::IsrPrime", loiterSpeed=best_loiter)
snapshot = interp.snapshot(sized, name="isrPrimeAsAnalyzed")
model.find("UavMissions").add(snapshot)

out = Path(tempfile.mkdtemp()) / "uav_missions_analyzed.sysml"
longeron.save(model, out)
back = longeron.Interpreter(longeron.load(out)).instantiate("UavMissions::isrPrimeAsAnalyzed")
assert math.isclose(back.slots["stationMinutes"], sized.slots["stationMinutes"], rel_tol=1e-12)
print(f"analyzed design point: loiterSpeed = {best_loiter:.1f} m/s")
print("saved text carries the computed values:\n")
print("    part isrPrimeAsAnalyzed" + out.read_text().split("part isrPrimeAsAnalyzed")[1][:170])
minutes = back.slots["stationMinutes"]
print(f"\nreloaded: {minutes:.1f} min (first-order body at {best_loiter:.1f} m/s)")

## The answer, and where the threads continue

No single mix reaches all three fronts. One winged VTOL base mix covers both
ISR and logistics. The intercept mission buys its own dart. The model now
carries the analyzed design point.

- T5 rebuilds the ISR winner as a population of named individuals and weighs
  them.
- T6 scores the fleet against stakeholder value and hunts for requirement
  violations.
- T7 measures the model's geometry claims with a CAD engine.